### Hardware Verification

#### Setup & Data Loading

In [31]:
import os
import numpy as np
import scipy.io as sio
data = sio.loadmat("Final11Pattern.mat")
Matrix = data["Matrix"]

# Givens rotation order [row_p, row_q, pivot_col]
GIVENS_PAIRS = [(0, 1, 0), (0, 2, 0), (1, 2, 1)]

# Parameter Setting
HW_DW = 18
HW_FRAC = 12
HW_ITERATION = 8    # Number of CORDIC iterations
PATTERN_ID = 8      # 0~10: select pattern from Final11Pattern.mat
EVD_MAX_ITER = 7    # Number of QR iterations


#### CORDIC_PE Hardware Verification

In [32]:
# ============================================================
# CORDIC_PE Hardware Bit-True Model
# ============================================================
HW_DW        = 18
HW_FRAC      = 12
HW_ITERATION = 8
HW_MASK      = (1 << HW_DW) - 1

def to_s18(v):
    """Wrap arbitrary integer to 18-bit two's complement."""
    v = int(v) & HW_MASK
    return v - (1 << HW_DW) if v >= (1 << (HW_DW - 1)) else v

def float_to_fix(f):
    """float -> Q5.12 signed 18-bit integer (truncate toward -inf)."""
    return to_s18(int(np.floor(f * (1 << HW_FRAC))))

def fix_to_float(i):
    """Q5.12 signed 18-bit integer -> float."""
    return i / (1 << HW_FRAC)

def to_hex18(v):
    """Signed integer -> hex string (2's complement, width follows HW_DW)."""
    n_digits = (HW_DW + 3) // 4
    return f"{int(v) & HW_MASK:0{n_digits}X}"

def csd_scale(v):
    """CSD 1/K approximation: (v>>>1)+(v>>>3) - ((v>>>6)+(v>>>9)), 18-bit."""
    a = to_s18((v >> 1) + (v >> 3))
    b = to_s18((v >> 6) + (v >> 9))
    return to_s18(a - b)

def hw_cordic_vectoring(in_x_fp, in_y_fp):
    """Bit-true Vectoring mode — mirrors CORDIC_PE.sv INITIAL_STAGE + VECTORING_CORE + OUTPUT_BLOCK."""
    in_x = float_to_fix(in_x_fp)
    in_y = float_to_fix(in_y_fp)

    flip = in_x < 0
    x = to_s18(-in_x) if flip else in_x
    y = to_s18(-in_y) if flip else in_y

    dir_bits = []
    for i in range(HW_ITERATION):
        dx, dy = y >> i, x >> i
        if y < 0:
            x, y = to_s18(x - dx), to_s18(y + dy)
            dir_bits.append(0)
        else:
            x, y = to_s18(x + dx), to_s18(y - dy)
            dir_bits.append(1)

    out_x = csd_scale(x)
    return fix_to_float(out_x), 0.0, {
        'flip': flip, 'dir_bits': dir_bits,
        'in_x': in_x, 'in_y': in_y,
        'out_x': out_x, 'out_y': 0,
    }

def hw_cordic_rotation(in_x_fp, in_y_fp, dir_info):
    """Bit-true Rotation mode — mirrors CORDIC_PE.sv INITIAL_STAGE + ROTATION_CORE + OUTPUT_BLOCK."""
    in_x = float_to_fix(in_x_fp)
    in_y = float_to_fix(in_y_fp)

    flip = dir_info['flip']
    x = to_s18(-in_x) if flip else in_x
    y = to_s18(-in_y) if flip else in_y

    for i, d in enumerate(dir_info['dir_bits']):
        dx, dy = y >> i, x >> i
        if d:
            x, y = to_s18(x + dx), to_s18(y - dy)
        else:
            x, y = to_s18(x - dx), to_s18(y + dy)

    ox, oy = csd_scale(x), csd_scale(y)
    return fix_to_float(ox), fix_to_float(oy), {
        'in_x': in_x, 'in_y': in_y,
        'out_x': ox, 'out_y': oy,
    }

In [33]:
# ============================================================
# Test Pattern Generation + Golden Computation + .dat Output
# ============================================================
import os

test_groups = []

# (A) Real matrix data: 11 patterns x 3 Givens pairs
for s in range(Matrix.shape[2]):
    M = Matrix[:, :, s].copy()
    for p, q, col in GIVENS_PAIRS:
        rot_cols = [m for m in range(col + 1, M.shape[1])]
        test_groups.append({
            'tag': f'pat{s}_G{p}{q}c{col}',
            'vec': (float(M[p, col]), float(M[q, col])),
            'rot': [(float(M[p, m]), float(M[q, m])) for m in rot_cols],
        })

# (B) Corner cases
corners = [
    ('y_zero',    ( 5.0,   0.0),  [( 2.0,  1.0)]),
    ('x_zero',    ( 0.0,   3.0),  [( 1.0,  1.0)]),
    ('x_neg',     (-4.0,   3.0),  [( 2.0, -1.0)]),
    ('both_neg',  (-5.0,  -5.0),  [(-3.0,  2.0)]),
    ('large',     (10.0,  10.0),  [(10.0, -10.0)]),
    ('tiny',      ( 0.000244140625, 0.000244140625), [(0.000244140625, 0.0)]),
    ('neg_large', (-10.0,  8.0),  [( 6.0, -7.0), ( 1.0,  1.0)]),
    ('one_one',   ( 1.0,   1.0),  [( 1.0, -1.0)]),
]
for tag, vec, rots in corners:
    test_groups.append({'tag': tag, 'vec': vec, 'rot': rots})

# (C) Random vectors
np.random.seed(2026)
for k in range(20):
    vx, vy = np.random.uniform(-10.0, 10.0, 2)
    n_rot = np.random.randint(1, 4)
    rots = list(zip(
        np.random.uniform(-10.0, 10.0, n_rot),
        np.random.uniform(-10.0, 10.0, n_rot),
    ))
    test_groups.append({
        'tag': f'rand{k}',
        'vec': (float(vx), float(vy)),
        'rot': [(float(a), float(b)) for a, b in rots],
    })

# Flatten to ordered operation stream + compute golden with HW bit-true model
ops = []
for g in test_groups:
    vx, vy = g['vec']
    _, _, dinfo = hw_cordic_vectoring(vx, vy)
    ops.append({
        'mode': 1,
        'in_x': dinfo['in_x'], 'in_y': dinfo['in_y'],
        'gold_x': dinfo['out_x'], 'gold_y': dinfo['out_y'],
        'tag': g['tag'],
    })
    for rx, ry in g['rot']:
        _, _, rinfo = hw_cordic_rotation(rx, ry, dinfo)
        ops.append({
            'mode': 0,
            'in_x': rinfo['in_x'], 'in_y': rinfo['in_y'],
            'gold_x': rinfo['out_x'], 'gold_y': rinfo['out_y'],
            'tag': g['tag'],
        })

# Write .dat files (hex, $readmemh compatible)
DAT_DIR = os.path.join('Verification_PE', '00_TESTBED', 'src')
os.makedirs(DAT_DIR, exist_ok=True)

dat_specs = {
    'PE_InMode.dat':  [f"{op['mode']}"          for op in ops],
    'PE_InX.dat':     [to_hex18(op['in_x'])      for op in ops],
    'PE_InY.dat':     [to_hex18(op['in_y'])      for op in ops],
    'PE_GoldenX.dat': [to_hex18(op['gold_x'])    for op in ops],
    'PE_GoldenY.dat': [to_hex18(op['gold_y'])    for op in ops],
}

for fname, lines in dat_specs.items():
    with open(os.path.join(DAT_DIR, fname), 'w') as f:
        f.write('\n'.join(lines) + '\n')

print(f"[OK] PE .dat x{len(dat_specs)} -> {DAT_DIR}/ ({len(ops)} ops)")

[OK] PE .dat x5 -> Verification_PE\00_TESTBED\src/ (165 ops)


#### EVD Hardware Verification

In [34]:
# ============================================================
# EVD Hardware Bit-True Model
# ============================================================

# PE Trace Infrastructure
PE_TRACE_EN = False
PE_TRACE    = []

PE_LABELS = {
    0: ("PE0", "ROW0_COL1", "G01"),
    1: ("PE1", "ROW0_COL2", "G02"),
    2: ("PE2", "ROW1_COL2", "G12"),
}

def pe_log(it, phase, pe_idx, mode, seq, in_x, in_y, out_x, out_y):
    """Append one trace entry when PE_TRACE_EN is True."""
    if not PE_TRACE_EN:
        return
    PE_TRACE.append({
        'it': it, 'phase': phase, 'pe_idx': pe_idx,
        'mode': mode, 'seq': seq,
        'in_x': in_x, 'in_y': in_y,
        'out_x': out_x, 'out_y': out_y,
    })

# HW EVD Functions 

def hw_apply_givens_rows(M, p, q, col, it=-1, pe_idx=-1, phase=None):
    """Apply one Givens rotation to matrix M (in-place).
    Vectoring on pivot (p,col)/(q,col), rotation on remaining columns."""
    _, _, dir_info = hw_cordic_vectoring(M[p, col], M[q, col])
    pe_log(it, phase, pe_idx, 'V', 0,
           dir_info['in_x'], dir_info['in_y'],
           dir_info['out_x'], dir_info['out_y'])
    M[p, col] = fix_to_float(dir_info['out_x'])
    M[q, col] = 0.0

    for seq, m in enumerate(range(col + 1, M.shape[1]), start=1):
        ox, oy, rinfo = hw_cordic_rotation(M[p, m], M[q, m], dir_info)
        pe_log(it, phase, pe_idx, 'R', seq,
               rinfo['in_x'], rinfo['in_y'],
               rinfo['out_x'], rinfo['out_y'])
        M[p, m] = ox
        M[q, m] = oy

    return M, dir_info


def hw_qrd(T, it=-1):
    """QR decomposition via 3 Givens rotations. Returns R and dir_infos."""
    R = T.copy()
    dir_infos = []
    for pe_idx, (p, q, col) in enumerate(GIVENS_PAIRS):
        R, di = hw_apply_givens_rows(R, p, q, col, it=it, pe_idx=pe_idx, phase='QRD')
        dir_infos.append(di)
    return R, dir_infos


def hw_rotation_pass(M, dir_infos, columns=False, it=-1, phase=None):
    """Replay rotation on all rows (columns=False) or all columns (columns=True)."""
    M = M.copy()
    for pe_idx, ((p, q, _col), dir_info) in enumerate(zip(GIVENS_PAIRS, dir_infos)):
        if columns:
            for seq, i in enumerate(range(M.shape[0]), start=1):
                ox, oy, rinfo = hw_cordic_rotation(M[i, p], M[i, q], dir_info)
                pe_log(it, phase, pe_idx, 'R', seq,
                       rinfo['in_x'], rinfo['in_y'],
                       rinfo['out_x'], rinfo['out_y'])
                M[i, p] = ox
                M[i, q] = oy
        else:
            for seq, m in enumerate(range(M.shape[1]), start=1):
                ox, oy, rinfo = hw_cordic_rotation(M[p, m], M[q, m], dir_info)
                pe_log(it, phase, pe_idx, 'R', seq,
                       rinfo['in_x'], rinfo['in_y'],
                       rinfo['out_x'], rinfo['out_y'])
                M[p, m] = ox
                M[q, m] = oy
    return M


def hw_iterative_qr_evd(A, max_iter=EVD_MAX_ITER):
    """Bit-true iterative QR EVD. Returns (eigenvalues, U, T_final)."""
    n = A.shape[0]
    T = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            T[i, j] = fix_to_float(float_to_fix(A[i, j]))

    U = np.eye(n)

    for it in range(max_iter):
        R, dir_infos = hw_qrd(T, it=it)
        RT = R.T.copy()
        T = hw_rotation_pass(RT, dir_infos, columns=False, it=it, phase='T-UPDATE')
        U = hw_rotation_pass(U, dir_infos, columns=True,  it=it, phase='U-UPDATE')

    lam = np.diag(T)
    return lam, U, T

In [35]:
# ============================================================
# EVD Golden Generation + .dat Output
# ============================================================

A = Matrix[:, :, PATTERN_ID].astype(float)
lam, U, T_final = hw_iterative_qr_evd(A, EVD_MAX_ITER)

DAT_DIR = os.path.join('00_TESTBED', 'src')
os.makedirs(DAT_DIR, exist_ok=True)

in_data_fix = [float_to_fix(A[i, j]) for i in range(3) for j in range(3)]
ev_fix = [float_to_fix(lam[i]) for i in range(3)]
u_fix = [float_to_fix(U[i, j]) for i in range(3) for j in range(3)]

dat_files = {
    'EVD_InData.dat':   in_data_fix,
    'EVD_GoldenEV.dat': ev_fix,
    'EVD_GoldenU.dat':  u_fix,
}

for fname, values in dat_files.items():
    with open(os.path.join(DAT_DIR, fname), 'w') as f:
        f.write('\n'.join(to_hex18(v) for v in values) + '\n')

print(f"[OK] EVD .dat x{len(dat_files)} -> {DAT_DIR}/ (pattern {PATTERN_ID})")

# Sanity check vs numpy reference
lam_ref, V_ref = np.linalg.eigh(A)

# Align order (match by closest eigenvalue)
idx = np.argsort(lam)
lam_sorted = lam[idx]
U_sorted = U[:, idx]

# Fix sign ambiguity
for i in range(3):
    if np.dot(V_ref[:, i], U_sorted[:, i]) < 0:
        U_sorted[:, i] *= -1

rmse_lam = np.sqrt(np.mean((lam_ref - lam_sorted) ** 2))
rmse_vec = np.sqrt(np.sum((V_ref - U_sorted) ** 2) / 9)

# Threshold relaxed to 5e-2: comparing HW bit-true (Q5.12 + CSD) vs float reference;
# the purpose is detecting gross bugs, not algorithm precision validation.
SANITY_THRESH = 5e-2
lam_ok = rmse_lam < SANITY_THRESH
vec_ok = rmse_vec < SANITY_THRESH
print(f"EVD pattern {PATTERN_ID}: RMSE lam={rmse_lam:.4f} vec={rmse_vec:.4f} -> "
      f"{'PASS' if lam_ok and vec_ok else 'FAIL'}")

[OK] EVD .dat x3 -> 00_TESTBED\src/ (pattern 8)
EVD pattern 8: RMSE lam=0.0389 vec=0.0153 -> PASS


#### PE Trace Debug Output

In [36]:
# ============================================================
# PE Trace Debug Output -> 00_TESTBED/PE_Trace.log
# ============================================================

PE_TRACE_EN = True
PE_TRACE.clear()

A_trace = Matrix[:, :, PATTERN_ID].astype(float)
_ = hw_iterative_qr_evd(A_trace, EVD_MAX_ITER)

PE_TRACE_EN = False

# Phase display order follows HW cnt sequence
PHASE_ORDER = [
    ('QRD',      'QRD (HW cnt 0~2)'),
    ('U-UPDATE', 'U-UPDATE (HW cnt 3~5)'),
    ('T-UPDATE', 'T-UPDATE (HW cnt 6~8)'),
]

lines = []
for it in range(EVD_MAX_ITER):
    lines.append(f"{'='*20} Iteration {it} {'='*20}")
    it_entries = [e for e in PE_TRACE if e['it'] == it]

    for phase_key, phase_label in PHASE_ORDER:
        phase_entries = [e for e in it_entries if e['phase'] == phase_key]
        if not phase_entries:
            continue
        lines.append(f"--- Phase: {phase_label} ---")
        for e in phase_entries:
            pe_name, rtl_name, givens = PE_LABELS[e['pe_idx']]
            mode_str = 'V   ' if e['mode'] == 'V' else f"R#{e['seq']}"
            ix_h, iy_h = to_hex18(e['in_x']), to_hex18(e['in_y'])
            ox_h, oy_h = to_hex18(e['out_x']), to_hex18(e['out_y'])
            lines.append(
                f"{pe_name} ({rtl_name}, {givens}) {mode_str:4s} "
                f"InX={ix_h} InY={iy_h} | OutX={ox_h} OutY={oy_h}"
            )
    lines.append("")

log_path = os.path.join('00_TESTBED', 'PE_Trace.log')
with open(log_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print(f"[OK] PE_Trace.log -> {log_path}")

[OK] PE_Trace.log -> 00_TESTBED\PE_Trace.log
